# Instagram Data Collection

Collects the caption and comments (with replies) of the Instagram posts listed in `instagram_post_links.txt`.

Steps:
1. Put one post link per line in `instagram_post_links.txt` (e.g. `https://www.instagram.com/p/XXXXXXXXXXX/` or `.../reel/XXXXXXXXXXX/`).
2. Set `INSTAGRAM_USERNAME` in the configuration cell.
3. Run the cells in order. The first time, you are asked for the cookies of a browser where you are logged in to Instagram. In Chrome on instagram.com: F12 > Network > reload the page > click the first request > Request Headers > right click `cookie` > Copy value. The session is then saved, so later runs do not ask again.

In [ ]:
import getpass
import itertools
import re
import time

import instaloader
import pandas as pd
import requests

## Configuration

In [ ]:
INSTAGRAM_USERNAME = "Chriss_163773"  # the password is asked for at login and never stored
LINKS_FILE = "instagram_post_links.txt"
OUTPUT_CSV = "InstagramComments.csv"
MAX_COMMENTS_PER_POST = 500  # top-level comments per post; replies of those comments are also collected
DELAY_BETWEEN_POSTS = 30  # seconds, to avoid being rate limited or flagged by Instagram

## Log in

In [ ]:
REQUIRED_COOKIES = ['sessionid', 'csrftoken', 'ds_user_id']

def parse_cookie_header(cookie_header):
    # 'name1=value1; name2=value2' -> {'name1': 'value1', 'name2': 'value2'}
    cookies = {}
    for part in cookie_header.split(';'):
        name, separator, value = part.strip().partition('=')
        if separator:
            cookies[name] = value
    return cookies

def login_with_browser_cookies(loader, username):
    # Reuse the session of a browser where you are already logged in (avoids Instagram's login checkpoint).
    # The cookies are typed into a prompt and are not stored in this notebook.
    cookies = parse_cookie_header(getpass.getpass('Paste the cookie request header: '))
    missing = [name for name in REQUIRED_COOKIES if name not in cookies]
    if missing:
        raise ValueError('Missing cookies: ' + ', '.join(missing) + '; copy the whole cookie header again')
    loader.load_session(username, cookies)
    loader.save_session_to_file()

def set_user_id(loader):
    # instaloader only learns the account id on a password login, but the comments endpoint needs it
    cookies = loader.save_session()
    if 'ds_user_id' not in cookies:
        raise ValueError('Saved session has no ds_user_id cookie; delete the session file and log in again')
    loader.context.user_id = cookies['ds_user_id']

def login(username):
    loader = instaloader.Instaloader()
    try:
        loader.load_session_from_file(username)
        print('Loaded saved session for', username)
    except FileNotFoundError:
        login_with_browser_cookies(loader, username)
        print('Logged in with browser cookies and saved session for', username)
    set_user_id(loader)
    return loader

if not INSTAGRAM_USERNAME:
    raise ValueError('Set INSTAGRAM_USERNAME in the configuration cell first')
loader = login(INSTAGRAM_USERNAME)

## Read post links

In [ ]:
SHORTCODE_PATTERN = re.compile(r'instagram\.com/(?:[^/]+/)?(?:p|reel|reels|tv)/([A-Za-z0-9_-]+)')

def shortcode_from_url(url):
    # The shortcode is the post id in the link, e.g. .../p/<shortcode>/
    match = SHORTCODE_PATTERN.search(url)
    if match is None:
        raise ValueError('Not an Instagram post link: ' + url)
    return match.group(1)

def read_post_links(path):
    # One link per line; blank lines and lines starting with # are ignored
    with open(path, 'r', encoding='utf-8') as links_file:
        lines = [line.strip() for line in links_file]
    return [line for line in lines if line and not line.startswith('#')]

post_links = read_post_links(LINKS_FILE)
len(post_links)

## Get captions and comments

In [ ]:
COMMENTS_PATH = 'api/v1/media/{media_id}/comments/'
REPLIES_PATH = 'api/v1/media/{media_id}/comments/{comment_id}/child_comments/'

def make_comments_session(loader):
    # The website's comments endpoint returns an HTML page unless the request has the website's own headers
    cookies = loader.save_session()
    session = requests.Session()
    session.cookies.update(cookies)
    session.headers.update({'User-Agent': loader.context.user_agent, 'Accept': '*/*',
                            'Referer': 'https://www.instagram.com/', 'X-CSRFToken': cookies['csrftoken'],
                            'X-IG-App-ID': '936619743392459', 'X-ASBD-ID': '129477', 'X-IG-WWW-Claim': '0',
                            'X-Requested-With': 'XMLHttpRequest', 'Sec-Fetch-Site': 'same-origin',
                            'Sec-Fetch-Mode': 'cors', 'Sec-Fetch-Dest': 'empty'})
    return session

def text_row(url, shortcode, source, username, created_at, likes, text):
    return {'post_url': url, 'shortcode': shortcode, 'source': source, 'username': username,
            'created_at': created_at, 'likes': likes, 'text': text}

def comment_row(url, shortcode, source, node):
    created_at = pd.to_datetime(node['created_at'], unit='s')
    return text_row(url, shortcode, source, node['user']['username'], created_at,
                    node.get('comment_like_count', 0), node['text'])

def iter_comment_nodes(loader, comments_session, media_id):
    # The website's endpoint; instaloader's mobile endpoint fails
    params = {'can_support_threading': 'true', 'permalink_enabled': 'false'}
    while True:
        page = loader.context.get_json(COMMENTS_PATH.format(media_id=media_id), params, session=comments_session)
        yield from page.get('comments', [])
        next_min_id = page.get('next_min_id')
        if not next_min_id:
            return
        params['min_id'] = next_min_id

def iter_reply_nodes(loader, comments_session, media_id, comment_node):
    # Replies included with the comment, or all pages of replies when only some of them are included
    preview = comment_node.get('preview_child_comments', [])
    if comment_node.get('child_comment_count', 0) == len(preview):
        yield from preview
        return
    replies_path = REPLIES_PATH.format(media_id=media_id, comment_id=comment_node['pk'])
    params = {'max_id': ''}
    while True:
        page = loader.context.get_json(replies_path, params, session=comments_session)
        yield from page.get('child_comments', [])
        if not page.get('has_more_head_child_comments'):
            return
        params = {'min_id': page['next_min_child_cursor']}

def get_post_texts(loader, comments_session, url, max_comments):
    shortcode = shortcode_from_url(url)
    post = instaloader.Post.from_shortcode(loader.context, shortcode)
    rows = []
    if post.caption:
        rows.append(text_row(url, shortcode, 'caption', post.owner_username, post.date_utc, post.likes, post.caption))
    for comment in itertools.islice(iter_comment_nodes(loader, comments_session, post.mediaid), max_comments):
        rows.append(comment_row(url, shortcode, 'comment', comment))
        for reply in iter_reply_nodes(loader, comments_session, post.mediaid, comment):
            rows.append(comment_row(url, shortcode, 'reply', reply))
    return rows

In [ ]:
comments_session = make_comments_session(loader)
all_post_texts = []
failed_links = []
for i, url in enumerate(post_links):
    print(i, '--------------- getting comments of', url, '------------------')
    try:
        post_texts = get_post_texts(loader, comments_session, url, MAX_COMMENTS_PER_POST)
        all_post_texts.extend(post_texts)
        print('got', len(post_texts), 'texts')
    except (instaloader.exceptions.InstaloaderException, ValueError) as err:
        print('failed:', err)
        failed_links.append(url)
    time.sleep(DELAY_BETWEEN_POSTS)

print('total texts:', len(all_post_texts), '| failed links:', len(failed_links))

In [ ]:
def clean_text(text):
    # same cleaning as the X data: remove #, commas, new lines and @mentions
    text = text.replace('#', '').replace(',', ' ').replace('\n', ' ')
    return re.sub(r'@[^\s]+', '', text).strip()

instagram_df = pd.DataFrame(all_post_texts)
instagram_df['text'] = instagram_df['text'].apply(clean_text)
instagram_df = instagram_df[instagram_df['text'] != '']

instagram_df.to_csv(OUTPUT_CSV, mode='w', header=True, index=True, encoding='utf-8')

instagram_df.head(900)